# 02 Train DistilBERT on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Git-Kapish/ReviewLens/blob/main/notebooks/02_train_distilbert_colab.ipynb)

Thin notebook: all logic is in `src/reviewlens/` and `scripts/train_transformer.py`.

**Before you run**
1. Runtime > Change runtime type > **T4 GPU**.
2. Upload the four split files from your local `data/processed/` (`train`, `val`, `test`, `timehold` `.parquet`, or `splits.zip`, about 14-16 MB) using the file-upload cell below. Using your local files guarantees the same splits as the baseline.
3. Run all cells. Training takes roughly 10-15 minutes on a T4.
4. The last cell downloads `distilbert_artifacts.zip`. Unzip it into your local repo root: it contains `models/distilbert/` (gitignored weights) and the files for `reports/` that get committed.

In [ ]:
!if [ ! -d "/content/ReviewLens" ]; then git clone https://github.com/Git-Kapish/ReviewLens.git /content/ReviewLens; else cd /content/ReviewLens && git pull; fi
%cd /content/ReviewLens
!pip install -q -e ".[transformer]"

In [ ]:
import torch

print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Data splits
Upload the four parquet files. If you skip the upload, the last line rebuilds them with `scripts/build_data.py` (streams the 2.7 GB source, about 10 minutes).

In [ ]:
import io
import os
from pathlib import Path
import zipfile

from google.colab import files

Path("data/processed").mkdir(parents=True, exist_ok=True)
if not all(Path(f"data/processed/{k}.parquet").exists() for k in ["train", "val", "test", "timehold"]):
    print("Upload the 4 parquet files (or splits.zip) from data/processed/ ...")
    for name, blob in files.upload().items():
        if name.endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(blob)) as z:
                z.extractall("data/processed")
        else:
            Path("data/processed", name).write_bytes(blob)
if not Path("data/processed/train.parquet").exists():
    !python scripts/build_data.py
print("Processed files in data/processed:", sorted(os.listdir("data/processed")))

## Train and evaluate
Writes `models/distilbert/`, `reports/transformer_metrics.json`, `reports/figures/transformer_curves.png` and `reports/predictions/`.

In [ ]:
!python scripts/train_transformer.py

In [ ]:
import json

from IPython.display import Image

m = json.load(open("reports/transformer_metrics.json"))
print("best epoch", m["best_epoch"], "| val macro-F1", round(m["val"]["macro_f1"], 4), "| test macro-F1", round(m["test"]["macro_f1"], 4))
Image("reports/figures/transformer_curves.png")

## Download artifacts

In [ ]:
import shutil

shutil.make_archive("distilbert_artifacts", "zip", root_dir=".", base_dir="models/distilbert")
!zip -qr distilbert_artifacts.zip reports/transformer_metrics.json reports/figures/transformer_curves.png reports/predictions
files.download("distilbert_artifacts.zip")